# 04 · Python이 빛나는 분석: 사업설명 텍스트  (모듈 4-B)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/04_text_analysis.ipynb)

> **STATA로는 사실상 못 하는** 분석. 수천 개 사업 설명문에서 "우리가 실제로 무엇을 하나"를
> 텍스트로 들여다본다. 이런 일(텍스트·수집·자동화·ML)은 **Python의 영역**이고 외부망에서 한다.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/sample_crs.csv")
print("사업 설명문 예시:")
for s in df["LongDescription"].head(3):
    print(" -", s)

## 1) 전체 상위 키워드 (TF-IDF)
어떤 표현이 포트폴리오 전반에서 두드러지나?

In [ ]:
tf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_df=0.6, max_features=15)
X = tf.fit_transform(df["LongDescription"])
scores = np.asarray(X.mean(axis=0)).ravel()
top = pd.Series(scores, index=tf.get_feature_names_out()).sort_values()
top.tail(10).plot.barh(figsize=(7, 4), title="전체 상위 키워드 (TF-IDF)")
plt.tight_layout(); plt.show()

## 2) 분야별 대표 키워드 — 텍스트가 드러내는 '사업의 실제 내용'
숫자가 아니라 **내용**을 분석한다. 분야마다 무엇을 하는지 키워드로 확인.

In [ ]:
def top_keywords(texts, k=3):
    v = TfidfVectorizer(stop_words="english", ngram_range=(2, 2))
    M = v.fit_transform(texts)
    s = np.asarray(M.mean(axis=0)).ravel()
    return ", ".join(pd.Series(s, index=v.get_feature_names_out()).sort_values().tail(k).index[::-1])

for sec, g in df.groupby("SectorName"):
    print(f"{sec:26s}: {top_keywords(g['LongDescription'])}")

---
✅ **포인트**: 이건 **Python만의 능력**이다(텍스트·수집·자동화·ML도 같은 결).
"이런 분석이 오면 Python, 외부망에서" 라고 판단하면 된다.

🧭 **인간 검증**: 키워드가 분야와 맞는지(예: Health→primary healthcare)는 **도메인 지식**으로 확인한다 —
AI도 사람도, 결과가 현장과 맞는지는 사람이 안다.